# Chapter 08 — 작은 데이터 분석 프로젝트 완성하기

Chapter 08 실습 결과를 과제 형식에 맞춰 정리한 Notebook이다.

`notebooks/ch08/00.ipynb`, `notebooks/ch08/01.ipynb`에서 수행한 분석을 바탕으로 **질문 → 전처리 → PK/FK → 병합 → EDA → 시각화 → 검증 → 해석 → 재현성** 순서로 정리했다.

- 제출 위치: `assignments/chapter08/chapter08.ipynb`
- 실습 Notebook: `notebooks/ch08/01.ipynb`
- 이미지 위치: `assignments/chapter08/images/`
- 금액성 분석 범위: `order_status == "completed"`
- 금액 계산 기준: `line_total = quantity × unit_price`


## 제출 정보

- GitHub ID: `cyw0927`
- 작성일: 2026-09-16
- 최종 Notebook URL: `https://github.com/cyw0927/-llm-data-analysis-course/blob/main/assignments/chapter08/chapter08.ipynb`
- 실습 Notebook: `notebooks/ch08/01.ipynb`


## 1. 프로젝트 질문

### 분석 질문
1. 카테고리별 completed 주문 기준 금액은 어떻게 다른가?
2. 월별 completed 주문 기준 금액과 주문 수는 어떻게 변하는가?
3. completed 주문 기준 고객별 구매 금액에는 어떤 차이가 있는가?

### 분석 범위
- 금액성 분석은 `order_status == "completed"` 주문만 사용한다.
- 주문 수는 주문 상세 행 수가 아니라 `order_id.nunique()` 기준으로 계산한다.
- 원본 CSV는 직접 수정하지 않고 전처리된 복사본을 사용한다.

### 사용할 데이터
- `data/raw/customers.csv`
- `data/raw/products.csv`
- `data/raw/orders.csv`
- `data/raw/order_items.csv`

### 계산 기준
- `order_status == "completed"` 적용
- `line_total = quantity × unit_price` 관계 확인
- 카테고리/월/고객별 금액은 동일한 completed source를 기준으로 비교

### 완료 기준
- 입력 데이터 구조 확인
- PK 결측/중복 점검
- FK 미매칭 점검
- 병합 전후 행 수와 미매칭 확인
- 핵심 EDA 3개 정리
- 대표 그래프 확인
- total consistency 확인
- 개인정보가 제거된 공개 고객 결과 확인
- 전체 프로젝트 재실행 가능 여부 확인


## 2. 실습 실행 기록

Chapter 08 실습은 `notebooks/ch08/01.ipynb`에서 수행했다.

실행 Notebook에는 코드 셀의 `execution_count`와 DataFrame/그래프 Output이 저장되어 있으며, 아래 내용은 해당 실행 결과를 기준으로 정리했다.

확인한 내용:
- `order_items`의 `line_total`을 `quantity × unit_price`로 계산
- completed 주문만 분리하여 금액성 분석 수행
- 월별 completed 주문 집계 생성
- 카테고리별 completed 주문 집계 생성
- 상품별/고객별 집계 및 시각화 수행


## 3. 입력 데이터와 전처리 검증

### 사용 원본 파일
- `customers.csv`
- `products.csv`
- `orders.csv`
- `order_items.csv`

### 확인 내용
- 각 데이터의 컬럼과 타입을 확인했다.
- 결측치와 중복 여부를 점검했다.
- `order_date`를 날짜형으로 변환해 월별 분석에 사용할 수 있게 했다.
- `quantity`, `unit_price`를 이용해 `line_total`을 계산했다.
- 원본을 직접 수정하지 않고 분석용 DataFrame을 별도로 만들어 사용했다.

### 나의 해석과 판단
분석 결과보다 먼저 데이터 구조와 계산 기준을 확인해야 이후 집계값을 신뢰할 수 있다고 판단했다.

### 한계와 추가 확인 사항
전처리 과정에서 값이 변환되거나 제외될 경우 정보 손실 가능성이 있으므로, 변환 실패 건수와 전후 행 수를 함께 확인하는 것이 필요하다.


## 4. PK/FK·병합 검증

### PK 점검
주요 PK:
- `customers.customer_id`
- `products.product_id`
- `orders.order_id`
- `order_items.order_item_id`

각 키의 결측/중복 여부를 확인했다.

### FK 점검
주요 FK:
- `orders.customer_id → customers.customer_id`
- `order_items.order_id → orders.order_id`
- `order_items.product_id → products.product_id`

### 병합 점검
병합 시 단순히 `merge()`만 실행하지 않고 관계와 행 수를 함께 확인했다.

- 주문 상세 → 주문: many-to-one
- 주문 상세 → 상품: many-to-one
- 고객별 집계 → 고객: one-to-one

### 나의 해석과 판단
오른쪽 테이블 키에 중복이 있으면 left merge라도 행 수가 증가할 수 있으므로, 병합 전후 행 수와 미매칭을 같이 확인해야 한다.


## 5. 핵심 EDA 결과

### 결과 1 — 카테고리별 completed 주문 기준 금액
실행 결과에서:
- 스포츠: **31,743,000**
- 전자기기: **26,400,000**

스포츠 카테고리가 가장 높은 금액을 기록했고 전자기기가 그 뒤를 이었다.

#### 나의 해석과 판단
카테고리 금액 차이는 확인되지만, 이것만으로 특정 카테고리의 선호도가 더 높다고 단정할 수는 없다.

#### 업무·분석적 의미
카테고리별 판매 수량과 평균 단가를 분리해서 보면 금액 차이가 수량 때문인지 단가 때문인지 더 구체적으로 확인할 수 있다.

#### 한계
프로모션, 재고, 계절성 등의 외부 요인은 현재 데이터만으로 확인할 수 없다.

---

### 결과 2 — 월별 completed 주문 기준 금액
실행 결과에서:
- 2025-07: **5,869,000 / 주문 8건**
- 2025-08: **15,621,000 / 주문 18건**

7월보다 8월에 completed 주문 금액과 주문 수가 모두 증가했다.

#### 나의 해석과 판단
월별 증감 자체는 확인할 수 있지만 증가 원인을 현재 데이터만으로 특정할 수는 없다.

#### 업무·분석적 의미
월별 주문 수와 평균 주문 금액을 분리해 보면 금액 변화가 주문량 변화 때문인지 주문당 금액 변화 때문인지 구분할 수 있다.

#### 한계
광고, 할인, 시즌 효과 등 원인 데이터가 없으므로 인과관계로 해석하지 않는다.

---

### 결과 3 — 고객별 completed 주문 구매 금액
고객별로 completed 주문을 묶어 구매 금액과 주문 횟수를 비교했다.

#### 나의 해석과 판단
고액 구매 고객과 반복 구매 고객은 같은 의미가 아니므로 `total_sales`, `order_count`, `avg_order_value`를 함께 봐야 한다.

#### 업무·분석적 의미
고객별 구매 패턴을 구분하면 향후 고객 세분화나 추가 분석 질문으로 연결할 수 있다.

#### 한계
현재 결과만으로 고객 충성도나 이탈 가능성을 판단할 수 없다.


## 6. Total consistency와 날짜 검증

같은 completed 주문 범위에서 만든 아래 집계는 같은 source total을 기준으로 비교해야 한다.

```text
completed source total
= category total
= monthly total
= customer total
```

또한 월별 분석에서는 `order_date` 변환 실패가 있으면 월별 금액에서 누락이 발생할 수 있으므로 completed 주문의 날짜 오류를 별도로 확인한다.

### 나의 판단
서로 다른 집계 결과가 같은 분석 범위를 사용했다면 최종 총합도 일치해야 한다. 불일치가 생기면 필터, 병합 행 증식, 날짜 오류, FK 미매칭 등을 먼저 확인해야 한다.


In [ ]:
# total consistency 재확인 코드

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "requirements.txt").exists() and (candidate / "data").exists():
        PROJECT_ROOT = candidate
        break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_sales_data
from src.preprocessing import preprocess_sales_data, validate_relationships
from src.midterm_project import (
    build_analysis_tables,
    build_key_duplicate_checks,
    build_project_validation,
)

raw_data = load_sales_data(PROJECT_ROOT / "data" / "raw")
processed_data = preprocess_sales_data(raw_data)
key_checks = build_key_duplicate_checks(processed_data)
relationship_checks = validate_relationships(processed_data).copy()
analysis_tables = build_analysis_tables(processed_data)

display(analysis_tables["total_consistency_check"])
display(analysis_tables["date_checks"])


## 7. 대표 시각화

실습에서 다음 시각화를 확인했다.

1. 카테고리별 completed 주문 기준 금액 — 막대그래프
2. 월별 completed 주문 기준 금액 — 선그래프
3. 상품/고객 관련 추가 시각화

### 그래프 선택 이유
카테고리 비교는 막대그래프가 항목 간 크기 비교에 적합하고, 월별 변화는 선그래프가 시간 흐름을 보기 쉽기 때문이다.

### 그래프와 원본 집계값 일치 여부
그래프는 앞서 생성한 집계 DataFrame을 사용해 작성했다.

### 그래프에서 직접 관찰한 사실
스포츠와 전자기기 카테고리의 completed 주문 기준 금액이 상대적으로 크게 나타났다. 월별 금액도 기간별 차이가 있었다.

### 그래프만으로 말할 수 없는 것
그래프만으로 프로모션 성공, 고객 선호, 수익성, 계절성 같은 원인을 단정할 수 없다.


## 8. 개인정보 검증

고객별 내부 분석에는 `customer_id`가 필요하지만 공개 결과에는 직접 식별정보를 남기지 않는 것을 원칙으로 한다.

공개 결과에서 제외할 컬럼:
- `customer_id`
- `name`
- `email`
- `phone`
- `address`

공개용 고객 라벨은 `Customer 01`, `Customer 02`처럼 순위 기반 익명 라벨을 사용한다.

### 나의 판단
분석에 필요한 식별 키와 공개 결과에 필요한 정보는 구분해야 한다. 내부 연결용 ID가 필요하더라도 제출용 결과에는 불필요한 개인정보를 제거하는 것이 적절하다.


## 9. 최종 Validation

Chapter 08 프로젝트의 핵심 검증 항목은 다음과 같다.

| 검증 항목 | 상태 | 확인 기준 |
| --- | --- | --- |
| pk_integrity | PASS 확인 대상 | PK 결측/중복 없음 |
| fk_integrity | PASS 확인 대상 | FK 미매칭 없음 |
| merge_checks_pass | PASS 확인 대상 | 병합 행 수/미매칭 정상 |
| line_total_consistency | PASS 확인 대상 | quantity × unit_price 일치 |
| completed_total_consistency | PASS 확인 대상 | category/month/customer 총합 일치 |
| category_sales_ratio_pct_sum | PASS 확인 대상 | 비율 합계 정상 |
| completed_rows_with_invalid_order_date | PASS 확인 대상 | 날짜 오류 없음 |
| public_customer_columns_safe | PASS 확인 대상 | 직접 식별정보 없음 |

최종 제출 전 아래 코드와 `reports/ch08_project_validation.csv`를 기준으로 다시 확인한다.


In [ ]:
project_validation = build_project_validation(
    key_checks,
    relationship_checks,
    analysis_tables,
)
display(project_validation)

assert project_validation["status"].eq("PASS").all(), "최종 Validation 실패"
print("Project Validation: PASS")


## 10. LLM 활용 기록

- 사용 여부: 예
- 사용 목적: 코드 설명과 검증 항목 확인, 문장 표현 보조
- Safe Context: 데이터 구조, 집계 결과, 코드, 검증 결과만 사용
- Prompt 요약: 분석 절차와 검증 항목에 빠진 부분이 있는지 확인
- 제안 요약: PK/FK, 병합, total consistency, 개인정보 검증 항목을 함께 확인
- 반영/수정/보류: 실제 실행 결과와 비교해 필요한 내용만 반영
- 사람이 검증한 근거: `notebooks/ch08/01.ipynb`의 실행 Output과 그래프, 원본 집계값을 직접 확인


## 11. 프로젝트 재현 확인

- 실습 Notebook 실행 여부: **완료**
- 실행 위치: `notebooks/ch08/01.ipynb`
- 전체 재실행 스크립트: `python scripts/run_midterm_project.py`
- Notebook과 핵심 분석 기준: completed 주문 기준으로 통일
- 생성 Evidence: CSV, 그래프, Validation 결과를 기준으로 확인

### 나의 해석과 판단
재현 가능한 프로젝트는 한 번 실행된 코드가 아니라, 같은 원본과 같은 기준에서 다시 실행했을 때 같은 결과를 얻을 수 있어야 한다. 따라서 코드뿐 아니라 필터 기준, 병합 관계, 계산식, 검증 조건을 함께 남기는 것이 중요하다.


## 12. 최종 프로젝트 요약

### 핵심 인사이트 3개
1. completed 주문 기준 카테고리별 금액에는 차이가 있으며, 스포츠와 전자기기 금액이 크게 나타났다.
2. 월별 completed 주문 금액과 주문 수는 기간별로 차이가 있었고, 2025년 7월보다 8월 값이 크게 증가했다.
3. 고객별 구매 금액은 고객마다 차이가 있으므로 총금액뿐 아니라 주문 횟수와 평균 주문 금액을 함께 보는 것이 필요하다.

### 가장 중요한 업무·분석적 의미
단순히 큰 숫자를 찾는 것보다, 동일한 분석 범위와 계산 기준을 유지하면서 결과를 교차 검증하는 과정이 중요하다.

### 현재 분석의 한계
- 프로모션, 광고, 재고, 계절성 데이터가 없다.
- completed 주문 기준 금액을 회계상 순매출이라고 단정할 수 없다.
- 집계 결과만으로 고객 행동의 원인을 설명할 수 없다.

### 다음 분석 제안
1. 카테고리별 판매 수량과 평균 단가를 분리해 비교
2. 월별 주문 수와 평균 주문 금액을 분리해 비교
3. 고객별 최근 구매일·구매 빈도·평균 구매 금액을 추가 분석


## 최종 체크

- [x] 질문과 지표를 연결했다.
- [x] 원본 데이터에서 분석을 시작했다.
- [x] PK/FK·병합 검증 항목을 포함했다.
- [x] `line_total = quantity × unit_price` 계산 기준을 사용했다.
- [x] 금액성 분석 범위를 completed 주문으로 고정했다.
- [x] category/month/customer total consistency 검증 코드를 포함했다.
- [x] 날짜 오류 검증 항목을 포함했다.
- [x] 공개 고객 결과의 직접 식별정보 제거 기준을 포함했다.
- [x] 대표 그래프와 원본 집계값의 관계를 설명했다.
- [x] 데이터에 없는 원인을 과도하게 단정하지 않았다.
- [x] LLM 활용 기록을 작성했다.
- [x] Chapter 08 실습 Notebook 실행을 완료했다.
- [x] 한계와 다음 분석을 작성했다.
- [x] 최종 Notebook URL을 작성했다.
